# Join de Datasets de Indicadores de Mercado

Notebook convertido a partir do script `join_datasets.py`.

## 1. Configuração de Ambiente e Caminhos



Definição de bibliotecas e paths de entrada/saída usados no processamento.

In [1]:
import pandas as pd

import numpy as np

from pathlib import Path



# Define paths (notebook-friendly)

notebook_dir = Path.cwd()

base_dir = notebook_dir.parent.parent.parent

soja3_path = base_dir / "datasets" / "datasets_indicadores" / "classificacao" / "SOJA3_tratado.csv"

cambio_path = base_dir / "indicadores_agro" / "indicador_agro_tratado" / "Cambio_USD_BRL.csv"

soja_chicago_path = base_dir / "indicadores_agro" / "indicador_agro_tratado" / "Soja_Chicago_USD_Bushel.csv"

output_path = notebook_dir / "indicadores_de_mercado_mais_indicadores_agro.csv"

## 2. Carregamento e Preparação dos Dados



Leitura dos arquivos CSV, padronização de datas e seleção das colunas de fechamento para os indicadores externos.

In [2]:
# Load datasets
print("Carregando datasets...")
soja3 = pd.read_csv(soja3_path)
cambio = pd.read_csv(cambio_path)
soja_chicago = pd.read_csv(soja_chicago_path)

# Standardize date column names and convert to datetime
soja3['Date'] = pd.to_datetime(soja3['Date'])
cambio['Data'] = pd.to_datetime(cambio['Data'])
soja_chicago['Data'] = pd.to_datetime(soja_chicago['Data'])

# Rename date columns for consistency
cambio = cambio.rename(columns={'Data': 'Date'})
soja_chicago = soja_chicago.rename(columns={'Data': 'Date'})

# Keep only 'close' from cambio and soja_chicago
cambio_close = cambio[['Date', 'close']].copy()
soja_chicago_close = soja_chicago[['Date', 'close']].copy()

# Rename close columns for clarity
cambio_close = cambio_close.rename(columns={'close': 'cambio_close'})
soja_chicago_close = soja_chicago_close.rename(columns={'close': 'soja_chicago_close'})

Carregando datasets...


## 3. Engenharia de Atributos e Fusão



Criação dos lags temporais dos indicadores de mercado, união com o dataset principal e inspeção inicial de qualidade.

In [3]:
# Create lags for cambio (-1, -3, -6, -10 days)
print("Criando lags para câmbio...")
for lag in [1, 3, 6, 10]:
    cambio_close[f'cambio_close_lag_{lag}d'] = cambio_close['cambio_close'].shift(lag)

# Create lags for soja chicago (-1, -3, -6, -10 days)
print("Criando lags para soja Chicago...")
for lag in [1, 3, 6, 10]:
    soja_chicago_close[f'soja_chicago_close_lag_{lag}d'] = soja_chicago_close['soja_chicago_close'].shift(lag)

# Merge all datasets
print("Unindo datasets...")
# First merge soja3 (left) with cambio (right)
merged = soja3.merge(cambio_close, on='Date', how='left')

# Then merge with soja_chicago (right)
merged = merged.merge(soja_chicago_close, on='Date', how='left')

# Sort by date
merged = merged.sort_values('Date').reset_index(drop=True)

# Display info ANTES de limpeza
print(f'\nResumo da fusão (ANTES da limpeza):')
print(f'  - Linhas SOJA3: {len(soja3)}')
print(f'  - Linhas Câmbio: {len(cambio)}')
print(f'  - Linhas Soja Chicago: {len(soja_chicago)}')
print(f'  - Linhas resultado: {len(merged)}')
print(f'  - NaNs totais: {merged.isna().sum().sum()}')

Criando lags para câmbio...
Criando lags para soja Chicago...
Unindo datasets...

Resumo da fusão (ANTES da limpeza):
  - Linhas SOJA3: 868
  - Linhas Câmbio: 1825
  - Linhas Soja Chicago: 1760
  - Linhas resultado: 868
  - NaNs totais: 125


## 4. Limpeza, Validação e Exportação



Aplicação dos critérios de completude das features críticas, análise final do resultado e salvamento do arquivo consolidado.

In [4]:
# Define critical features (features usadas no treinamento)
critical_features = ['Close', 'Low', 'High', 'Open',
                     'OBV', 'FWMA', 'TEMA', 'HLC3', 'BB_upper', 'BB_middle', 'BB_lower',
                     'cambio_close_lag_1d', 'cambio_close_lag_3d', 'cambio_close_lag_6d', 'cambio_close_lag_10d',
                     'soja_chicago_close_lag_1d', 'soja_chicago_close_lag_3d', 'soja_chicago_close_lag_6d', 'soja_chicago_close_lag_10d']

# Identificar linhas incompletas
incomplete_rows = merged[critical_features].isna().any(axis=1)
print(f'\nLinhas incompletas (com NaN em features críticas): {incomplete_rows.sum()}')
if incomplete_rows.sum() > 0:
    print(f'  Datas afetadas: {merged[incomplete_rows]["Date"].dt.strftime("%Y-%m-%d").tolist()}')

# Limpar dataset (remover linhas com NaN em features críticas)
print('\nLimpando dataset (dropna em features críticas)...')
merged_clean = merged[~incomplete_rows].reset_index(drop=True)

# Display sample
print(f'\nAmostra das primeiras linhas com novos dados de mercado:')
market_cols = ['Date'] + [col for col in merged_clean.columns if 'cambio' in col or 'soja_chicago' in col]
print(merged_clean[market_cols].head(15))

# Display info DEPOIS de limpeza
print(f'\nResumo APÓS limpeza:')
print(f'  - Linhas antes: {len(merged)}')
print(f'  - Linhas depois: {len(merged_clean)}')
print(f'  - Linhas removidas: {len(merged) - len(merged_clean)} ({(len(merged) - len(merged_clean))/len(merged)*100:.1f}%)')
print(f'  - NaNs totais: {merged_clean.isna().sum().sum()}')

# Save output
print(f'\nSalvando resultado em: {output_path}')
merged_clean.to_csv(output_path, index=False)
print('✓ Concluído!')


Linhas incompletas (com NaN em features críticas): 25
  Datas afetadas: ['2021-05-31', '2021-07-05', '2021-09-06', '2021-11-25', '2022-01-17', '2022-02-21', '2022-05-30', '2022-06-20', '2022-07-04', '2022-09-05', '2022-11-24', '2022-12-26', '2023-01-02', '2023-01-16', '2023-05-29', '2023-06-19', '2023-07-04', '2023-09-04', '2023-11-23', '2024-01-15', '2024-02-19', '2024-05-27', '2024-06-19', '2024-07-04', '2024-09-02']

Limpando dataset (dropna em features críticas)...

Amostra das primeiras linhas com novos dados de mercado:
         Date  cambio_close  cambio_close_lag_1d  cambio_close_lag_3d  \
0  2021-05-27        5.3105               5.3307               5.3618   
1  2021-05-28        5.2382               5.3105               5.3185   
2  2021-06-01        5.2166               5.2380               5.3105   
3  2021-06-02        5.1497               5.2166               5.2382   
4  2021-06-04        5.0763               5.0745               5.2166   
5  2021-06-07        5.0446  